# **1. Setup & Libraries**

In [ ]:
!pip install matplotlib-scalebar
!pip install contextily

# For post-hoc seasonal tests
!pip install scikit-posthocs

!pip install pymannkendall


In [ ]:
!pip install lightgbm


                                              0.0/1.5 MB ? eta -:--:--
                                              0.0/1.5 MB 1.3 MB/s eta 0:00:02
     --                                       0.1/1.5 MB 1.0 MB/s eta 0:00:02
     ------                                   0.2/1.5 MB 1.7 MB/s eta 0:00:01
     ---------                                0.4/1.5 MB 2.0 MB/s eta 0:00:01
     ----------------                         0.6/1.5 MB 2.6 MB/s eta 0:00:01
     ----------------------                   0.8/1.5 MB 3.2 MB/s eta 0:00:01
     ------------------------------------     1.3/1.5 MB 4.5 MB/s eta 0:00:01
     ---------------------------------------  1.4/1.5 MB 4.0 MB/s eta 0:00:01
     ---------------------------------------- 1.5/1.5 MB 3.8 MB/s eta 0:00:00


In [ ]:
import os
import glob
import calendar

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.ticker import ScalarFormatter
from matplotlib_scalebar.scalebar import ScaleBar
import seaborn as sns

from scipy.stats import mannwhitneyu, kruskal, wilcoxon
import pymannkendall as mk
import scikit_posthocs as sp


# **2. Load Dataset**

In [ ]:
file_path = r"C:\Users\pc\Documents\Rainfall\Data\Final processed 3-hourly rainfall data.csv"
df_rain = pd.read_csv(file_path)
df_rain

,StationID,StationName,Latitude,Longitude,Primary_Region,Secondary_Region,Datetime,Year,Month,Day,Time,Season,DewPointTemperature,StationLevelPressure,SP,DR,Humidity,Rainfall
0,41977,Ambagan(Ctg.),22.3500,91.8167,South East,Coastal,2003-01-01 00:00:00,2003,1,1,0,Winter,12.0,1009.7,0.0,0.0,91.0,0.0
1,41977,Ambagan(Ctg.),22.3500,91.8167,South East,Coastal,2003-01-01 03:00:00,2003,1,1,3,Winter,13.0,1011.3,0.0,0.0,74.0,0.0
2,41977,Ambagan(Ctg.),22.3500,91.8167,South East,Coastal,2003-01-01 06:00:00,2003,1,1,6,Winter,15.0,1011.2,4.0,31.0,57.0,0.0
3,41977,Ambagan(Ctg.),22.3500,91.8167,South East,Coastal,2003-01-01 09:00:00,2003,1,1,9,Winter,9.0,1010.3,2.0,5.0,35.0,0.0
4,41977,Ambagan(Ctg.),22.3500,91.8167,South East,Coastal,2003-01-01 12:00:00,2003,1,1,12,Winter,13.0,1010.4,0.0,0.0,61.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2116614,11929,Teknaf,20.8667,92.3000,South East,Coastal,2023-07-31 09:00:00,2023,7,31,9,Monsoon,26.0,1016.0,1.0,10.0,73.0,0.0
2116615,11929,Teknaf,20.8667,92.3000,South East,Coastal,2023-07-31 12:00:00,2023,7,31,12,Monsoon,26.0,1016.0,0.0,0.0,82.0,0.0
2116616,11929,Teknaf,20.8667,92.3000,South East,Coastal,2023-07-31 15:00:00,2023,7,31,15,Monsoon,26.0,1016.0,0.0,0.0,88.0,0.0
2116617,11929,Teknaf,20.8667,92.3000,South East,Coastal,2023-07-31 18:00:00,2023,7,31,18,Monsoon,26.0,1016.0,0.0,0.0,88.0,0.0


In [ ]:
# REMOVE THE LATE STATION (Example ID: 10325)
# Replace 10325 with the actual ID of the 2008 station
stations_to_drop = [11921]
df_rain = df_rain[~df_rain["StationID"].isin(stations_to_drop)].reset_index(drop=True)
df_rain


,StationID,StationName,Latitude,Longitude,Primary_Region,Secondary_Region,Datetime,Year,Month,Day,Time,Season,DewPointTemperature,StationLevelPressure,SP,DR,Humidity,Rainfall
0,41977,Ambagan(Ctg.),22.3500,91.8167,South East,Coastal,2003-01-01 00:00:00,2003,1,1,0,Winter,12.0,1009.7,0.0,0.0,91.0,0.0
1,41977,Ambagan(Ctg.),22.3500,91.8167,South East,Coastal,2003-01-01 03:00:00,2003,1,1,3,Winter,13.0,1011.3,0.0,0.0,74.0,0.0
2,41977,Ambagan(Ctg.),22.3500,91.8167,South East,Coastal,2003-01-01 06:00:00,2003,1,1,6,Winter,15.0,1011.2,4.0,31.0,57.0,0.0
3,41977,Ambagan(Ctg.),22.3500,91.8167,South East,Coastal,2003-01-01 09:00:00,2003,1,1,9,Winter,9.0,1010.3,2.0,5.0,35.0,0.0
4,41977,Ambagan(Ctg.),22.3500,91.8167,South East,Coastal,2003-01-01 12:00:00,2003,1,1,12,Winter,13.0,1010.4,0.0,0.0,61.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2069862,11929,Teknaf,20.8667,92.3000,South East,Coastal,2023-07-31 09:00:00,2023,7,31,9,Monsoon,26.0,1016.0,1.0,10.0,73.0,0.0
2069863,11929,Teknaf,20.8667,92.3000,South East,Coastal,2023-07-31 12:00:00,2023,7,31,12,Monsoon,26.0,1016.0,0.0,0.0,82.0,0.0
2069864,11929,Teknaf,20.8667,92.3000,South East,Coastal,2023-07-31 15:00:00,2023,7,31,15,Monsoon,26.0,1016.0,0.0,0.0,88.0,0.0
2069865,11929,Teknaf,20.8667,92.3000,South East,Coastal,2023-07-31 18:00:00,2023,7,31,18,Monsoon,26.0,1016.0,0.0,0.0,88.0,0.0


# **Hyperparameter Training**

In [ ]:
# ============================================================
# STATION-WISE LIGHTGBM BASELINE - HYPERPARAMETER TUNING
# ============================================================
#
# - Assumes df_rain (DataFrame) is ALREADY LOADED in memory.
# - Uses same preprocessing, splits, and metrics as other baselines.
# - Builds station-wise LightGBM models:
#     * Quantile regression (rainfall) per (station, quantile)
#     * Binary classifiers per station for flash, peak, acc
# - Tuning objective: validation CRPS_log
# - Saves in TUNING_SAVE_DIR:
#     * lgbm_tuning_results.csv  (all trials)
#     * best_trial.json          (best hyperparameters)
#     * best_models.pkl          (dict of LightGBM models)
# ============================================================

import os, random, math, json
import numpy as np
import pandas as pd
import lightgbm as lgb
import joblib

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import average_precision_score, roc_auc_score

# ------------------------------------------------------------
# 0) Reproducibility + Device
# ------------------------------------------------------------
def set_seed(seed=42, deterministic=False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42, deterministic=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ------------------------------------------------------------
# 0.5) Paths (EDIT THIS IF NEEDED)
# ------------------------------------------------------------
TUNING_SAVE_DIR = r"C:\Users\pc\Documents\Rainfall\Baselines\LightGBM\Tuning_v1"
os.makedirs(TUNING_SAVE_DIR, exist_ok=True)
print("Tuning directory:", TUNING_SAVE_DIR)

# ============================================================
# 1) Core utilities (same style as other baselines)
# ============================================================

class NaNIgnoringStandardScaler:
    def __init__(self, eps=1e-8):
        self.eps = eps
        self.mean_ = None
        self.std_ = None

    def fit(self, X):
        self.mean_ = np.nanmean(X, axis=0)
        self.std_  = np.nanstd(X, axis=0)
        self.std_  = np.where(self.std_ < self.eps, 1.0, self.std_)
        return self

    def transform(self, X):
        return (X - self.mean_) / self.std_


def pinball_loss(y_hat_q, y_true, q, mask=None):
    e = y_true - y_hat_q
    loss = torch.maximum((q - 1.0) * e, q * e)
    if mask is not None:
        loss = loss * mask
        denom = mask.sum().clamp_min(1.0)
        return loss.sum() / denom
    return loss.mean()


def crps_from_quantiles(y_hat, y_true, quantiles, mask=None):
    losses = []
    for k, q in enumerate(quantiles):
        losses.append(pinball_loss(y_hat[..., k], y_true, q, mask=mask))
    return 2.0 * torch.stack(losses).mean()


def brier_score(probs, targets, mask=None):
    loss = (probs - targets) ** 2
    if mask is not None:
        loss = loss * mask
        denom = mask.sum().clamp_min(1.0)
        return loss.sum() / denom
    return loss.mean()


def safe_div(a, b, eps=1e-12):
    return a / (b + eps)


def event_metrics_binary(probs, y_true, mask, thresholds=(0.1, 0.2, 0.3, 0.5)):
    valid = mask > 0.5
    if valid.sum() < 50:
        return {
            "n_valid": int(valid.sum()),
            "pr_auc": np.nan,
            "roc_auc": np.nan,
            "by_thr": {thr: {"POD": np.nan, "FAR": np.nan, "CSI": np.nan} for thr in thresholds},
        }

    p = probs[valid]
    y = y_true[valid]

    pr_auc = average_precision_score(y, p) if (y.max() > 0 and y.min() < 1) else np.nan
    try:
        roc_auc = roc_auc_score(y, p) if (y.max() > 0 and y.min() < 1) else np.nan
    except Exception:
        roc_auc = np.nan

    by_thr = {}
    for thr in thresholds:
        yhat = (p >= thr).astype(np.float32)
        TP = ((yhat == 1) & (y == 1)).sum()
        FP = ((yhat == 1) & (y == 0)).sum()
        FN = ((yhat == 0) & (y == 1)).sum()

        POD = safe_div(TP, TP + FN)
        FAR = safe_div(FP, TP + FP)
        CSI = safe_div(TP, TP + FP + FN)

        by_thr[thr] = {"POD": float(POD), "FAR": float(FAR), "CSI": float(CSI)}

    return {
        "n_valid": int(valid.sum()),
        "pr_auc": float(pr_auc) if pr_auc == pr_auc else np.nan,
        "roc_auc": float(roc_auc) if roc_auc == roc_auc else np.nan,
        "by_thr": by_thr,
    }


def make_splits(T, train_frac=0.7, val_frac=0.15):
    train_end = int(T * train_frac)
    val_end   = int(T * (train_frac + val_frac))
    return train_end, val_end


def scale_inputs_train_only(X_raw, train_end):
    T, N, F = X_raw.shape
    X_flat = X_raw.reshape(T*N, F)
    train_flat = X_flat[:train_end*N]
    scaler = NaNIgnoringStandardScaler().fit(train_flat)
    X_scaled = scaler.transform(X_flat).reshape(T, N, F).astype(np.float32)
    return X_scaled, scaler


def compute_thresholds_train(Y, M, train_end, q=0.95):
    Y_train = Y[:train_end]
    M_train = M[:train_end]
    N = Y.shape[1]
    thr = np.zeros(N, dtype=np.float32)

    global_vals = Y_train[M_train > 0.5]
    global_fallback = np.nanpercentile(global_vals, q*100) if global_vals.size > 0 else 0.0

    for j in range(N):
        vals = Y_train[:, j][M_train[:, j] > 0.5]
        if len(vals) < 100:
            thr[j] = global_fallback
        else:
            thr[j] = np.nanpercentile(vals, q*100)
    return thr


def compute_acc24(Y_rain, M_y, H_out=8):
    T, N = Y_rain.shape
    Acc  = np.full((T, N), np.nan, dtype=np.float32)
    Mask = np.zeros((T, N), dtype=np.float32)
    for t in range(T - H_out):
        window = Y_rain[t+1:t+1+H_out]
        wmask  = M_y[t+1:t+1+H_out]
        ok = (wmask.sum(axis=0) == H_out)
        Mask[t, ok] = 1.0
        Acc[t, ok]  = window[:, ok].sum(axis=0)
    return Acc, Mask

# ============================================================
# 2) Preprocessing: full grid, station/time features
# ============================================================

def prepare_full_grid(df_rain):
    df = df_rain.copy()
    df["Datetime"] = pd.to_datetime(df["Datetime"])
    df = df.sort_values(["Datetime", "StationID"]).reset_index(drop=True)

    # DR -> sin/cos
    dr = df["DR"].astype(np.float32).to_numpy()
    dr_rad = np.deg2rad(dr)
    df["DR_sin"] = np.sin(dr_rad)
    df["DR_cos"] = np.cos(dr_rad)

    stations = (
        df[["StationID", "StationName", "Latitude", "Longitude"]]
        .drop_duplicates()
        .sort_values("StationID")
        .reset_index(drop=True)
    )
    stations["node_index"] = np.arange(len(stations))
    N = len(stations)
    print("Total stations:", N)

    times = np.sort(df["Datetime"].unique())
    T = len(times)
    print("Total unique times:", T)

    full_index = pd.MultiIndex.from_product(
        [times, stations["StationID"].values],
        names=["Datetime", "StationID"]
    )

    meteo_cols = [
        "DewPointTemperature",
        "StationLevelPressure",
        "SP",
        "Humidity",
        "Rainfall",
        "DR_sin",
        "DR_cos",
    ]

    df2 = df.set_index(["Datetime", "StationID"])[meteo_cols].sort_index()
    df_full = df2.reindex(full_index)
    X_raw = df_full.values.reshape(T, N, len(meteo_cols)).astype(np.float32)

    M_in = (~np.isnan(X_raw)).astype(np.float32)

    rain_idx = meteo_cols.index("Rainfall")
    Y_rain = X_raw[..., rain_idx]
    M_y    = M_in[..., rain_idx]

    # time-of-day / month features
    dt_index = pd.DatetimeIndex(times)
    hour = dt_index.hour.astype(np.float32)
    month = dt_index.month.astype(np.float32)

    hour_sin = np.sin(2*np.pi*(hour/24.0))
    hour_cos = np.cos(2*np.pi*(hour/24.0))
    month_sin = np.sin(2*np.pi*((month-1)/12.0))
    month_cos = np.cos(2*np.pi*((month-1)/12.0))

    time_feats = np.stack([hour_sin, hour_cos, month_sin, month_cos], axis=-1).astype(np.float32)
    time_feats = np.repeat(time_feats[:, None, :], N, axis=1)

    # regime ids (season)
    season_by_time = (
        df.groupby("Datetime")["Season"]
          .agg(lambda s: s.mode().iloc[0] if len(s.mode()) else s.iloc[0])
          .reindex(times)
          .astype(str)
          .values
    )
    unique_seasons = sorted(pd.unique(season_by_time))
    season_to_id = {s: i for i, s in enumerate(unique_seasons)}
    season_ids = np.array([season_to_id[s] for s in season_by_time], dtype=np.int64)

    return {
        "stations": stations,
        "times": times,
        "X_raw": X_raw,
        "M_in": M_in,
        "Y_rain": Y_rain,
        "M_y": M_y,
        "meteo_cols": meteo_cols,
        "time_feats": time_feats,
        "season_ids": season_ids,
        "unique_seasons": unique_seasons,
    }

# ============================================================
# 3) Dataset (same structure as NN baselines)
# ============================================================

class BanglaRainDataset(Dataset):
    def __init__(
        self,
        X_scaled, M_in, time_feats,
        Y_rain, M_y,
        Acc24, MaskAcc24,
        season_ids,
        thr3h, thrAcc24,
        t_start, t_end,
        T_in=16, H_out=8,
        peak_min_obs=None,
    ):
        self.X_scaled = X_scaled
        self.M_in = M_in
        self.time_feats = time_feats
        self.Y_rain = Y_rain
        self.M_y = M_y
        self.Acc24 = Acc24
        self.MaskAcc24 = MaskAcc24
        self.season_ids = season_ids
        self.thr3h = thr3h
        self.thrAcc24 = thrAcc24
        self.T_in = T_in
        self.H_out = H_out
        self.peak_min_obs = peak_min_obs if peak_min_obs is not None else (H_out // 2)

        self.indices = []
        for t in range(t_start + (T_in - 1), t_end - H_out):
            self.indices.append(t)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        t = self.indices[idx]

        x  = self.X_scaled[t-self.T_in+1:t+1]     # [T_in,N,F]
        m  = self.M_in[t-self.T_in+1:t+1]         # [T_in,N,F]
        tf = self.time_feats[t-self.T_in+1:t+1]   # [T_in,N,4]

        x = np.nan_to_num(x, nan=0.0).astype(np.float32)
        x_all = np.concatenate([x, m.astype(np.float32), tf], axis=-1)  # [T_in,N,F+F+4]

        y  = self.Y_rain[t+1:t+1+self.H_out]       # [H_out,N]
        my = self.M_y[t+1:t+1+self.H_out].astype(np.float32)
        y_log = np.log1p(np.nan_to_num(y, nan=0.0)).astype(np.float32)

        # flash at t+1
        y_next = self.Y_rain[t+1]
        m_next = self.M_y[t+1].astype(np.float32)
        flash  = ((y_next >= self.thr3h) & (m_next > 0.5)).astype(np.float32)
        mflash = m_next.copy()

        # peak24
        y_win = self.Y_rain[t+1:t+1+self.H_out]
        m_win = self.M_y[t+1:t+1+self.H_out].astype(np.float32)
        mpeak = (m_win.sum(axis=0) >= self.peak_min_obs).astype(np.float32)
        y_peak = np.nanmax(np.where(m_win > 0.5, y_win, np.nan), axis=0)
        peak24 = ((y_peak >= self.thr3h) & (mpeak > 0.5)).astype(np.float32)

        # acc24
        acc  = self.Acc24[t]
        macc = self.MaskAcc24[t].astype(np.float32)
        acc24 = ((acc >= self.thrAcc24) & (macc > 0.5)).astype(np.float32)

        regime_id = int(self.season_ids[t])

        return (
            torch.from_numpy(x_all),                         # x
            torch.tensor(regime_id, dtype=torch.long),       # regime
            torch.from_numpy(y_log),                         # y_log
            torch.from_numpy(my),                            # my
            torch.from_numpy(flash), torch.from_numpy(mflash),
            torch.from_numpy(peak24), torch.from_numpy(mpeak),
            torch.from_numpy(acc24),  torch.from_numpy(macc),
        )

# ============================================================
# 4) Evaluation function (same as NN baselines)
# ============================================================

@torch.no_grad()
def evaluate(model, loader, quantiles, thresholds=(0.1, 0.2, 0.3, 0.5)):
    model.eval()

    total_crps_log = 0.0
    total_crps_mm  = 0.0
    total_brier_flash = 0.0
    total_brier_peak  = 0.0
    total_brier_acc   = 0.0
    nb = 0

    H_out = None
    sum_abs = None
    sum_sq  = None
    count   = None

    qs = np.array(list(quantiles), dtype=np.float32)
    k50 = int(np.argmin(np.abs(qs - 0.5)))

    flash_p_list, flash_y_list, flash_m_list = [], [], []
    peak_p_list,  peak_y_list,  peak_m_list  = [], [], []
    acc_p_list,   acc_y_list,   acc_m_list   = [], [], []

    for (x, reg, y_log, my, flash, mflash, peak, mpeak, acc, macc) in loader:
        x = x.to(device)
        reg = reg.to(device)
        y_log = y_log.to(device)
        my = my.to(device)
        flash = flash.to(device); mflash = mflash.to(device)
        peak  = peak.to(device);  mpeak  = mpeak.to(device)
        acc   = acc.to(device);   macc   = macc.to(device)
        q_hat, flash_logits, peak_logits, acc_logits = model(x, reg)

        # CRPS (log space)
        total_crps_log += crps_from_quantiles(q_hat, y_log, quantiles, mask=my).item()

        # Convert to mm and compute CRPS
        q_mm = torch.expm1(q_hat).clamp_min(0.0)
        y_mm = torch.expm1(y_log).clamp_min(0.0)
        total_crps_mm  += crps_from_quantiles(q_mm, y_mm, quantiles, mask=my).item()

        # Risk Brier
        p_flash = torch.sigmoid(flash_logits)
        p_peak  = torch.sigmoid(peak_logits)
        p_acc   = torch.sigmoid(acc_logits)

        total_brier_flash += brier_score(p_flash, flash, mask=mflash).item()
        total_brier_peak  += brier_score(p_peak,  peak,  mask=mpeak ).item()
        total_brier_acc   += brier_score(p_acc,   acc,   mask=macc  ).item()

        nb += 1

        # Median quantile
        q50_log = q_hat[..., k50]
        q50_mm  = torch.expm1(q50_log).clamp_min(0.0)

        B, H, N = y_mm.shape
        if H_out is None:
            H_out = H
            sum_abs = torch.zeros(H_out, device="cpu")
            sum_sq  = torch.zeros(H_out, device="cpu")
            count   = torch.zeros(H_out, device="cpu")

        for h in range(H):
            m = (my[:, h, :] > 0.5)
            if m.any():
                err = (q50_mm[:, h, :][m] - y_mm[:, h, :][m]).detach().cpu()
                sum_abs[h] += err.abs().sum()
                sum_sq[h]  += (err**2).sum()
                count[h]   += m.sum().detach().cpu()

        flash_p_list.append(p_flash.detach().cpu().reshape(-1))
        flash_y_list.append(flash.detach().cpu().reshape(-1))
        flash_m_list.append(mflash.detach().cpu().reshape(-1))

        peak_p_list.append(p_peak.detach().cpu().reshape(-1))
        peak_y_list.append(peak.detach().cpu().reshape(-1))
        peak_m_list.append(mpeak.detach().cpu().reshape(-1))

        acc_p_list.append(p_acc.detach().cpu().reshape(-1))
        acc_y_list.append(acc.detach().cpu().reshape(-1))
        acc_m_list.append(macc.detach().cpu().reshape(-1))

    out = {
        "CRPS_log":   total_crps_log / max(nb, 1),
        "CRPS_mm":    total_crps_mm  / max(nb, 1),
        "Brier_flash": total_brier_flash / max(nb, 1),
        "Brier_peak":  total_brier_peak  / max(nb, 1),
        "Brier_acc":   total_brier_acc   / max(nb, 1),
    }

    # MAE / RMSE by lead
    maes, rmses, counts = [], [], []
    for h in range(H_out):
        if count[h].item() < 1:
            maes.append(np.nan); rmses.append(np.nan); counts.append(0)
        else:
            maes.append(float((sum_abs[h] / count[h]).item()))
            rmses.append(float(torch.sqrt(sum_sq[h] / count[h]).item()))
            counts.append(int(count[h].item()))
    out.update({"MAE_mm_by_lead": maes, "RMSE_mm_by_lead": rmses, "n_valid_by_lead": counts})

    # Event metrics
    flash_p = torch.cat(flash_p_list).numpy()
    flash_y = torch.cat(flash_y_list).numpy()
    flash_m = torch.cat(flash_m_list).numpy()

    peak_p  = torch.cat(peak_p_list).numpy()
    peak_y  = torch.cat(peak_y_list).numpy()
    peak_m  = torch.cat(peak_m_list).numpy()

    acc_p   = torch.cat(acc_p_list).numpy()
    acc_y   = torch.cat(acc_y_list).numpy()
    acc_m   = torch.cat(acc_m_list).numpy()

    out["Flash_metrics"]  = event_metrics_binary(flash_p, flash_y, flash_m, thresholds=thresholds)
    out["Peak24_metrics"] = event_metrics_binary(peak_p,  peak_y,  peak_m,  thresholds=thresholds)
    out["Acc24_metrics"]  = event_metrics_binary(acc_p,   acc_y,   acc_m,   thresholds=thresholds)

    return out

# ============================================================
# 5) Build data objects + loaders
# ============================================================

def build_data_objects(df_rain, T_in=16, H_out=8, train_frac=0.7, val_frac=0.15):
    prep = prepare_full_grid(df_rain)
    T = len(prep["times"])
    train_end, val_end = make_splits(T, train_frac=train_frac, val_frac=val_frac)

    X_scaled, scaler = scale_inputs_train_only(prep["X_raw"], train_end)

    thr3h = compute_thresholds_train(prep["Y_rain"], prep["M_y"], train_end, q=0.95)

    Acc24, MaskAcc24 = compute_acc24(prep["Y_rain"], prep["M_y"], H_out=H_out)
    thrAcc24 = compute_thresholds_train(Acc24, MaskAcc24, train_end, q=0.95)

    ds_train = BanglaRainDataset(
        X_scaled, prep["M_in"], prep["time_feats"],
        prep["Y_rain"], prep["M_y"],
        Acc24, MaskAcc24,
        prep["season_ids"],
        thr3h, thrAcc24,
        t_start=0, t_end=train_end,
        T_in=T_in, H_out=H_out
    )
    ds_val = BanglaRainDataset(
        X_scaled, prep["M_in"], prep["time_feats"],
        prep["Y_rain"], prep["M_y"],
        Acc24, MaskAcc24,
        prep["season_ids"],
        thr3h, thrAcc24,
        t_start=train_end, t_end=val_end,
        T_in=T_in, H_out=H_out
    )
    ds_test = BanglaRainDataset(
        X_scaled, prep["M_in"], prep["time_feats"],
        prep["Y_rain"], prep["M_y"],
        Acc24, MaskAcc24,
        prep["season_ids"],
        thr3h, thrAcc24,
        t_start=val_end, t_end=T,
        T_in=T_in, H_out=H_out
    )

    return prep, scaler, (thr3h, thrAcc24), (Acc24, MaskAcc24), (train_end, val_end, T), (ds_train, ds_val, ds_test)


def make_loaders(ds_train, ds_val, ds_test, batch_size=256):
    train_loader = DataLoader(ds_train, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader   = DataLoader(ds_val,   batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(ds_test,  batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader

# ============================================================
# 6) Flatten loader -> tabular data for LightGBM
# ============================================================

def build_tabular_from_loader(loader):
    """
    Build two tabular datasets:
      - Regression: per (time, lead, station) rainfall in mm.
      - Classification: per (time, station) flash/peak/acc.
    """
    # Regression
    X_reg_list = []
    y_reg_list = []
    station_reg_list = []
    lead_reg_list = []
    mask_reg_list = []

    # Classification
    X_clf_list = []
    station_clf_list = []
    flash_y_list, flash_m_list = [], []
    peak_y_list,  peak_m_list  = [], []
    acc_y_list,   acc_m_list   = [], []

    for (x, reg, y_log, my, flash, mflash, peak, mpeak, acc, macc) in loader:
        x_np = x[:, -1, :, :].numpy()  # [B,N,F_all] last step
        y_mm = torch.expm1(y_log).clamp_min(0.0).numpy()  # [B,H,N]
        my_np = my.numpy()

        flash_np = flash.numpy()
        mflash_np = mflash.numpy()
        peak_np  = peak.numpy()
        mpeak_np = mpeak.numpy()
        acc_np   = acc.numpy()
        macc_np  = macc.numpy()

        B, H, N = y_mm.shape

        # --- Regression rows: (time, lead, station) ---
        for b in range(B):
            for h in range(H):
                mask_h = my_np[b, h, :] > 0.5
                if not mask_h.any():
                    continue
                X_bh = x_np[b]  # [N,F]
                y_bh = y_mm[b, h, :]
                for j in range(N):
                    if mask_h[j]:
                        X_reg_list.append(X_bh[j])
                        y_reg_list.append(y_bh[j])
                        station_reg_list.append(j)
                        lead_reg_list.append(h)
                        mask_reg_list.append(1.0)

        # --- Classification rows: (time, station) ---
        for b in range(B):
            for j in range(N):
                if mflash_np[b, j] > 0.5:
                    X_clf_list.append(x_np[b, j])
                    station_clf_list.append(j)
                    flash_y_list.append(flash_np[b, j])
                    peak_y_list.append(peak_np[b, j])
                    acc_y_list.append(acc_np[b, j])
                    flash_m_list.append(mflash_np[b, j])
                    peak_m_list.append(mpeak_np[b, j])
                    acc_m_list.append(macc_np[b, j])

    X_reg = np.array(X_reg_list, dtype=np.float32)
    y_reg = np.array(y_reg_list, dtype=np.float32)
    station_reg = np.array(station_reg_list, dtype=np.int64)
    lead_reg = np.array(lead_reg_list, dtype=np.int64)
    mask_reg = np.array(mask_reg_list, dtype=np.float32)

    X_clf = np.array(X_clf_list, dtype=np.float32)
    station_clf = np.array(station_clf_list, dtype=np.int64)
    flash_y = np.array(flash_y_list, dtype=np.float32)
    flash_m = np.array(flash_m_list, dtype=np.float32)
    peak_y  = np.array(peak_y_list, dtype=np.float32)
    peak_m  = np.array(peak_m_list, dtype=np.float32)
    acc_y   = np.array(acc_y_list, dtype=np.float32)
    acc_m   = np.array(acc_m_list, dtype=np.float32)

    return {
        "X_reg": X_reg,
        "y_reg": y_reg,
        "station_reg": station_reg,
        "lead_reg": lead_reg,
        "mask_reg": mask_reg,

        "X_clf": X_clf,
        "station_clf": station_clf,
        "flash_y": flash_y,
        "flash_m": flash_m,
        "peak_y": peak_y,
        "peak_m": peak_m,
        "acc_y": acc_y,
        "acc_m": acc_m,
    }

# ============================================================
# 7) Station-wise LightGBM model wrapper
# ============================================================

class StationWiseLGBMBaseline(nn.Module):
    """
    LightGBM station-wise baseline:
      - One quantile regressor per (station, quantile).
      - One binary classifier per station for flash / peak / acc.
    Wrapped as nn.Module so `evaluate()` works unchanged.
    """
    def __init__(self,
                 lgbm_regressors,
                 lgbm_flash,
                 lgbm_peak,
                 lgbm_acc,
                 num_stations,
                 H_out,
                 quantiles):
        super().__init__()
        self.lgbm_regressors = lgbm_regressors
        self.lgbm_flash = lgbm_flash
        self.lgbm_peak = lgbm_peak
        self.lgbm_acc = lgbm_acc
        self.N = num_stations
        self.H_out = H_out
        self.quantiles = list(quantiles)
        self.K = len(self.quantiles)

    def forward(self, x, regime_id):
        B, T, N, F = x.shape
        assert N == self.N

        x_last = x[:, -1, :, :].detach().cpu().numpy()  # [B,N,F]

        q_mm = np.zeros((B, self.H_out, N, self.K), dtype=np.float32)
        flash_prob = np.zeros((B, N), dtype=np.float32)
        peak_prob  = np.zeros((B, N), dtype=np.float32)
        acc_prob   = np.zeros((B, N), dtype=np.float32)

        for j in range(N):
            X_j = x_last[:, j, :]  # [B,F]

            # ------- Quantile regressors (unchanged) -------
            for k, q in enumerate(self.quantiles):
                model = self.lgbm_regressors.get((j, q), None)
                if model is None:
                    yhat = np.zeros(B, dtype=np.float32)
                else:
                    yhat = model.predict(X_j)
                q_mm[:, :, j, k] = yhat[:, None]  # same for all H_out

            # ------- Binary classifiers: use predict() -------
            m_flash = self.lgbm_flash.get(j, None)
            if m_flash is None:
                flash_prob[:, j] = 0.0
            else:
                # Booster.predict -> probabilities in [0,1] for objective="binary"
                p = m_flash.predict(X_j)
                # handle 1D or 2D just in case
                flash_prob[:, j] = p if p.ndim == 1 else p[:, 1]

            m_peak = self.lgbm_peak.get(j, None)
            if m_peak is None:
                peak_prob[:, j] = 0.0
            else:
                p = m_peak.predict(X_j)
                peak_prob[:, j] = p if p.ndim == 1 else p[:, 1]

            m_acc = self.lgbm_acc.get(j, None)
            if m_acc is None:
                acc_prob[:, j] = 0.0
            else:
                p = m_acc.predict(X_j)
                acc_prob[:, j] = p if p.ndim == 1 else p[:, 1]

        q_mm_tensor = torch.from_numpy(q_mm).to(device)
        q_log = torch.log1p(q_mm_tensor).clamp_min(0.0)

        def prob_to_logits(p):
            p = torch.clamp(p, 1e-6, 1.0 - 1e-6)
            return torch.log(p / (1.0 - p))

        flash_logits = prob_to_logits(torch.from_numpy(flash_prob).to(device))
        peak_logits  = prob_to_logits(torch.from_numpy(peak_prob).to(device))
        acc_logits   = prob_to_logits(torch.from_numpy(acc_prob).to(device))

        return q_log, flash_logits, peak_logits, acc_logits

# ============================================================
# 8) LightGBM training helpers
# ============================================================

def sample_hparams(rng):
    """
    Random-search hyperparameters for LightGBM.

    - learning_rate: log-uniform in [1e-3, 3e-1]
    - n_estimators: 100–800
    - num_leaves: 16–256
    - min_data_in_leaf: 20–200
    - feature_fraction, bagging_fraction: 0.5–1.0
    - bagging_freq: 1–7
    - lambda_l2: log-uniform in [1e-4, 1e2]
    """
    log_lr_min = math.log10(1e-3)
    log_lr_max = math.log10(3e-1)
    learning_rate = 10 ** rng.uniform(log_lr_min, log_lr_max)

    num_leaves = int(round(2 ** rng.uniform(4, 8)))  # ~16 – ~256

    return {
        "learning_rate":    float(learning_rate),
        "n_estimators":     int(rng.randint(100, 801)),   # 100–800
        "num_leaves":       max(16, min(256, num_leaves)),
        "min_data_in_leaf": int(rng.randint(20, 201)),    # 20–200
        "feature_fraction": float(rng.uniform(0.5, 1.0)), # 0.5–1.0
        "bagging_fraction": float(rng.uniform(0.5, 1.0)), # 0.5–1.0
        "bagging_freq":     int(rng.randint(1, 8)),       # 1–7
        "lambda_l2":        float(10 ** rng.uniform(-4, 2)),  # 1e-4 – 1e2
    }


def train_station_regressors(X_reg, y_reg, station_reg, params_base,quantiles, num_stations):
    lgbm_regressors = {}
    for j in range(num_stations):
        mask_j = (station_reg == j)
        if not np.any(mask_j):
            continue
        X_j = X_reg[mask_j]
        y_j = y_reg[mask_j]
        if X_j.shape[0] < 50:
            continue

        for q in quantiles:
            params = params_base.copy()
            params.update({
                "objective": "quantile",
                "alpha": q,
                "metric": "quantile",
            })
            dtrain = lgb.Dataset(X_j, label=y_j)
            model = lgb.train(
                params,
                dtrain,
                num_boost_round=params["n_estimators"]
            )
            lgbm_regressors[(j, q)] = model
    return lgbm_regressors


def train_station_classifiers(tab, num_stations, params_base):
    X_clf = tab["X_clf"]
    station_clf = tab["station_clf"]

    def train_one_target(y, m):
        models = {}
        for j in range(num_stations):
            mask_station = (station_clf == j)
            if not np.any(mask_station):
                continue

            X_j = X_clf[mask_station]
            y_j = y[mask_station]
            m_j = m[mask_station]

            # require valid label
            valid = (m_j > 0.5)
            X_j = X_j[valid]
            y_j = y_j[valid]

            if X_j.shape[0] < 20:
                # dummy model predicting all zeros
                X_dummy = np.zeros((50, X_clf.shape[1]), dtype=np.float32)
                y_dummy = np.zeros(50, dtype=np.float32)
                params = params_base.copy()
                params.update({
                    "objective": "binary",
                    "metric": "binary_logloss",
                })
                dtrain = lgb.Dataset(X_dummy, label=y_dummy)
                model = lgb.train(
                    params,
                    dtrain,
                    num_boost_round=10
                )
                models[j] = model
                continue

            params = params_base.copy()
            params.update({
                "objective": "binary",
                "metric": "binary_logloss",
            })
            dtrain = lgb.Dataset(X_j, label=y_j)
            model = lgb.train(
                params,
                dtrain,
                num_boost_round=params["n_estimators"]
            )
            models[j] = model
        return models

    lgbm_flash = train_one_target(tab["flash_y"], tab["flash_m"])
    lgbm_peak  = train_one_target(tab["peak_y"],  tab["peak_m"])
    lgbm_acc   = train_one_target(tab["acc_y"],   tab["acc_m"])

    return lgbm_flash, lgbm_peak, lgbm_acc

# ============================================================
# 9) Master tuning function (standalone)
# ============================================================

def tune_lightgbm_stationwise(
    df_rain,
    T_in=16,
    H_out=8,
    train_frac=0.7,
    val_frac=0.15,
    quantiles=(0.1, 0.5, 0.9),
    n_trials=10,
):
    prep, scaler, (thr3h, thrAcc24), (Acc24, MaskAcc24), splits, datasets = build_data_objects(
        df_rain,
        T_in=T_in,
        H_out=H_out,
        train_frac=train_frac,
        val_frac=val_frac,
    )
    train_end, val_end, T_total = splits
    ds_train, ds_val, ds_test = datasets

    train_loader, val_loader, _ = make_loaders(
        ds_train, ds_val, ds_test, batch_size=256
    )

    N_stations = len(prep["stations"])
    print("N_stations:", N_stations)

    print("Building tabular TRAIN data for LightGBM...")
    tab_train = build_tabular_from_loader(train_loader)
    print("  X_reg:", tab_train["X_reg"].shape,
          "| y_reg:", tab_train["y_reg"].shape)
    print("  X_clf:", tab_train["X_clf"].shape)

    rng = np.random.RandomState(42)

    best_score = float("inf")
    best_trial = None
    best_models = None
    trial_rows = []

    for trial in range(1, n_trials + 1):
        print(f"\n[LightGBM Tuning] Trial {trial}/{n_trials}")
        hp = sample_hparams(rng)
        print("  Hyperparams:", hp)

        params_base = {
            "boosting_type":      "gbdt",
            "num_leaves":         hp["num_leaves"],
            "min_data_in_leaf":   hp["min_data_in_leaf"],
            "feature_fraction":   hp["feature_fraction"],
            "bagging_fraction":   hp["bagging_fraction"],
            "bagging_freq":       hp["bagging_freq"],
            "lambda_l2":          hp["lambda_l2"],
            "learning_rate":      hp["learning_rate"],
            "verbosity":          -1,
            "seed":               42,
            "feature_fraction_seed": 42,
            "bagging_seed":       42,
            "deterministic":      True,
            "force_row_wise":     True,
            "num_threads":        -1,
            "n_estimators":       hp["n_estimators"],
        }

        # Train regressors
        lgbm_reg = train_station_regressors(
            tab_train["X_reg"],
            tab_train["y_reg"],
            tab_train["station_reg"],
            params_base,
            quantiles=quantiles,
            num_stations=N_stations,
        )

        # Train classifiers
        lgbm_flash, lgbm_peak, lgbm_acc = train_station_classifiers(
            tab_train,
            num_stations=N_stations,
            params_base=params_base,
        )

        # Wrap as PyTorch module
        model = StationWiseLGBMBaseline(
            lgbm_regressors=lgbm_reg,
            lgbm_flash=lgbm_flash,
            lgbm_peak=lgbm_peak,
            lgbm_acc=lgbm_acc,
            num_stations=N_stations,
            H_out=H_out,
            quantiles=quantiles,
        ).to(device)

        # Evaluate on validation set
        val_scores = evaluate(model, val_loader, quantiles=quantiles)
        val_CRPS_log = float(val_scores["CRPS_log"])
        print(f"  -> val_CRPS_log = {val_CRPS_log:.4f}")

        row = {
            "trial": trial,
            "val_CRPS_log": val_CRPS_log,
        }
        row.update(hp)
        trial_rows.append(row)

        # Save all trials so far
        pd.DataFrame(trial_rows).to_csv(
            os.path.join(TUNING_SAVE_DIR, "lgbm_tuning_results.csv"),
            index=False,
        )

        # Track best trial
        if val_CRPS_log < best_score - 1e-6:
            best_score = val_CRPS_log
            best_trial = row
            best_models = {
                "lgbm_reg": lgbm_reg,
                "lgbm_flash": lgbm_flash,
                "lgbm_peak": lgbm_peak,
                "lgbm_acc": lgbm_acc,
            }
            print(f"  -> New best trial (CRPS_log = {best_score:.4f})")

            # Save best trial + models
            with open(os.path.join(TUNING_SAVE_DIR, "best_trial.json"), "w") as f:
                json.dump(best_trial, f, indent=2)

            joblib.dump(best_models, os.path.join(TUNING_SAVE_DIR, "best_models.pkl"))

    print("\n[LightGBM Tuning] Finished.")
    print("Best val_CRPS_log:", best_score)
    print("Best trial:", best_trial)
    return best_trial

# ============================================================
# 10) RUN TUNING (CALL THIS AFTER df_rain IS LOADED)
# ============================================================
# Example:
best_trial = tune_lightgbm_stationwise(
    df_rain=df_rain,
    T_in=16,
    H_out=8,
    train_frac=0.7,
    val_frac=0.15,
    quantiles=(0.1, 0.5, 0.9),
    n_trials=15,
)
print("\nBest LightGBM trial:")
print(best_trial)


Using device: cpu
Tuning directory: C:\Users\pc\Documents\Rainfall\Baselines\LightGBM\Tuning_v1
Total stations: 34
Total unique times: 61360
N_stations: 34
Building tabular TRAIN data for LightGBM...


C:\Users\pc\AppData\Local\Temp\ipykernel_9604\1629955347.py:341: RuntimeWarning: All-NaN slice encountered
  y_peak = np.nanmax(np.where(m_win > 0.5, y_win, np.nan), axis=0)


  X_reg: (11538022, 18) | y_reg: (11538022,)
  X_clf: (1442245, 18)

[LightGBM Tuning] Trial 1/15
  Hyperparams: {'learning_rate': 0.008468008575248332, 'n_estimators': 206, 'num_leaves': 223, 'min_data_in_leaf': 91, 'feature_fraction': 0.7993292420985183, 'bagging_fraction': 0.5780093202212182, 'bagging_freq': 3, 'lambda_l2': 0.00039796923010559877}
  -> val_CRPS_log = 0.1411
  -> New best trial (CRPS_log = 0.1411)

[LightGBM Tuning] Trial 2/15
  Hyperparams: {'learning_rate': 0.013728250383906291, 'n_estimators': 763, 'num_leaves': 40, 'min_data_in_leaf': 150, 'feature_fraction': 0.5102922471479012, 'bagging_fraction': 0.9849549260809971, 'bagging_freq': 4, 'lambda_l2': 42.78743516759823}


C:\Users\pc\AppData\Local\Temp\ipykernel_9604\1629955347.py:341: RuntimeWarning: All-NaN slice encountered
  y_peak = np.nanmax(np.where(m_win > 0.5, y_win, np.nan), axis=0)


  -> val_CRPS_log = 0.1387
  -> New best trial (CRPS_log = 0.1387)

[LightGBM Tuning] Trial 3/15
  Hyperparams: {'learning_rate': 0.0010044517908654516, 'n_estimators': 260, 'num_leaves': 251, 'min_data_in_leaf': 77, 'feature_fraction': 0.762378215816119, 'bagging_fraction': 0.7159725093210578, 'bagging_freq': 1, 'lambda_l2': 0.14081468939305825}


...

C:\Users\pc\AppData\Local\Temp\ipykernel_9604\1629955347.py:341: RuntimeWarning: All-NaN slice encountered
  y_peak = np.nanmax(np.where(m_win > 0.5, y_win, np.nan), axis=0)


  -> val_CRPS_log = nan

[LightGBM Tuning] Trial 15/15
  Hyperparams: {'learning_rate': 0.01225433607687274, 'n_estimators': 340, 'num_leaves': 28, 'min_data_in_leaf': 71, 'feature_fraction': 0.7816377859881918, 'bagging_fraction': 0.8477580432130638, 'bagging_freq': 5, 'lambda_l2': 0.0031315069138816223}


C:\Users\pc\AppData\Local\Temp\ipykernel_9604\1629955347.py:341: RuntimeWarning: All-NaN slice encountered
  y_peak = np.nanmax(np.where(m_win > 0.5, y_win, np.nan), axis=0)


  -> val_CRPS_log = 0.1387


PermissionError: [Errno 13] Permission denied: 'C:\\Users\\pc\\Documents\\Rainfall\\Baselines\\LightGBM\\Tuning_v1\\lgbm_tuning_results.csv'

# **Model Training**

In [ ]:
# ============================================================
# STATION-WISE LIGHTGBM BASELINE - FINAL TRAINING (Option A)
# ============================================================
#
# - Assumes df_rain (DataFrame) is ALREADY LOADED in memory.
# - Assumes tuning has been run and produced:
#       TUNING_SAVE_DIR/best_trial.json
# - This script:
#       * Rebuilds preprocessing & splits
#       * Builds tabular TRAIN+VAL data
#       * Trains station-wise LightGBM models with best hyperparams
#       * Saves final models + scaler + thresholds, etc. to:
#           FINAL_SAVE_DIR/final_lightgbm_models.pkl
# - No evaluation on TEST here (pure training).
# ============================================================

import os, random, math, json
import numpy as np
import pandas as pd
import lightgbm as lgb
import joblib

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import average_precision_score, roc_auc_score

# ------------------------------------------------------------
# 0) Reproducibility + Device
# ------------------------------------------------------------
def set_seed(seed=42, deterministic=False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cnn.benchmark = False

set_seed(42, deterministic=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ------------------------------------------------------------
# 0.5) Paths (EDIT THESE IF NEEDED)
# ------------------------------------------------------------
TUNING_SAVE_DIR = r"C:\Users\pc\Documents\Rainfall\Baselines\LightGBM\Tuning_v1"
FINAL_SAVE_DIR  = r"C:\Users\pc\Documents\Rainfall\Baselines\LightGBM\Final_Run\v1"

os.makedirs(TUNING_SAVE_DIR, exist_ok=True)
os.makedirs(FINAL_SAVE_DIR, exist_ok=True)

print("Tuning directory :", TUNING_SAVE_DIR)
print("Final save dir   :", FINAL_SAVE_DIR)

BEST_TRIAL_PATH = os.path.join(TUNING_SAVE_DIR, "best_trial.json")

# ============================================================
# 1) Core utilities  (same as tuning)
# ============================================================

class NaNIgnoringStandardScaler:
    def __init__(self, eps=1e-8):
        self.eps = eps
        self.mean_ = None
        self.std_ = None

    def fit(self, X):
        self.mean_ = np.nanmean(X, axis=0)
        self.std_  = np.nanstd(X, axis=0)
        self.std_  = np.where(self.std_ < self.eps, 1.0, self.std_)
        return self

    def transform(self, X):
        return (X - self.mean_) / self.std_

def pinball_loss(y_hat_q, y_true, q, mask=None):
    e = y_true - y_hat_q
    loss = torch.maximum((q - 1.0) * e, q * e)
    if mask is not None:
        loss = loss * mask
        denom = mask.sum().clamp_min(1.0)
        return loss.sum() / denom
    return loss.mean()

def crps_from_quantiles(y_hat, y_true, quantiles, mask=None):
    losses = []
    for k, q in enumerate(quantiles):
        losses.append(pinball_loss(y_hat[..., k], y_true, q, mask=mask))
    return 2.0 * torch.stack(losses).mean()

def brier_score(probs, targets, mask=None):
    loss = (probs - targets) ** 2
    if mask is not None:
        loss = loss * mask
        denom = mask.sum().clamp_min(1.0)
        return loss.sum() / denom
    return loss.mean()

def safe_div(a, b, eps=1e-12):
    return a / (b + eps)

def event_metrics_binary(probs, y_true, mask, thresholds=(0.1, 0.2, 0.3, 0.5)):
    valid = mask > 0.5
    if valid.sum() < 50:
        return {
            "n_valid": int(valid.sum()),
            "pr_auc": np.nan,
            "roc_auc": np.nan,
            "by_thr": {thr: {"POD": np.nan, "FAR": np.nan, "CSI": np.nan} for thr in thresholds},
        }

    p = probs[valid]
    y = y_true[valid]

    pr_auc = average_precision_score(y, p) if (y.max() > 0 and y.min() < 1) else np.nan
    try:
        roc_auc = roc_auc_score(y, p) if (y.max() > 0 and y.min() < 1) else np.nan
    except Exception:
        roc_auc = np.nan

    by_thr = {}
    for thr in thresholds:
        yhat = (p >= thr).astype(np.float32)
        TP = ((yhat == 1) & (y == 1)).sum()
        FP = ((yhat == 1) & (y == 0)).sum()
        FN = ((yhat == 0) & (y == 1)).sum()

        POD = safe_div(TP, TP + FN)
        FAR = safe_div(FP, TP + FP)
        CSI = safe_div(TP, TP + FP + FN)

        by_thr[thr] = {"POD": float(POD), "FAR": float(FAR), "CSI": float(CSI)}

    return {
        "n_valid": int(valid.sum()),
        "pr_auc": float(pr_auc) if pr_auc == pr_auc else np.nan,
        "roc_auc": float(roc_auc) if roc_auc == roc_auc else np.nan,
        "by_thr": by_thr,
    }

def make_splits(T, train_frac=0.7, val_frac=0.15):
    train_end = int(T * train_frac)
    val_end   = int(T * (train_frac + val_frac))
    return train_end, val_end

def scale_inputs_train_only(X_raw, train_end):
    T, N, F = X_raw.shape
    X_flat = X_raw.reshape(T*N, F)
    train_flat = X_flat[:train_end*N]
    scaler = NaNIgnoringStandardScaler().fit(train_flat)
    X_scaled = scaler.transform(X_flat).reshape(T, N, F).astype(np.float32)
    return X_scaled, scaler

def compute_thresholds_train(Y, M, train_end, q=0.95):
    Y_train = Y[:train_end]
    M_train = M[:train_end]
    N = Y.shape[1]
    thr = np.zeros(N, dtype=np.float32)

    global_vals = Y_train[M_train > 0.5]
    global_fallback = np.nanpercentile(global_vals, q*100) if global_vals.size > 0 else 0.0

    for j in range(N):
        vals = Y_train[:, j][M_train[:, j] > 0.5]
        if len(vals) < 100:
            thr[j] = global_fallback
        else:
            thr[j] = np.nanpercentile(vals, q*100)
    return thr

def compute_acc24(Y_rain, M_y, H_out=8):
    T, N = Y_rain.shape
    Acc  = np.full((T, N), np.nan, dtype=np.float32)
    Mask = np.zeros((T, N), dtype=np.float32)
    for t in range(T - H_out):
        window = Y_rain[t+1:t+1+H_out]
        wmask  = M_y[t+1:t+1+H_out]
        ok = (wmask.sum(axis=0) == H_out)
        Mask[t, ok] = 1.0
        Acc[t, ok]  = window[:, ok].sum(axis=0)
    return Acc, Mask

# ============================================================
# 2) Preprocessing: full grid, station/time features
# ============================================================

def prepare_full_grid(df_rain):
    df = df_rain.copy()
    df["Datetime"] = pd.to_datetime(df["Datetime"])
    df = df.sort_values(["Datetime", "StationID"]).reset_index(drop=True)

    # DR -> sin/cos
    dr = df["DR"].astype(np.float32).to_numpy()
    dr_rad = np.deg2rad(dr)
    df["DR_sin"] = np.sin(dr_rad)
    df["DR_cos"] = np.cos(dr_rad)

    stations = (
        df[["StationID", "StationName", "Latitude", "Longitude"]]
        .drop_duplicates()
        .sort_values("StationID")
        .reset_index(drop=True)
    )
    stations["node_index"] = np.arange(len(stations))
    N = len(stations)
    print("Total stations:", N)

    times = np.sort(df["Datetime"].unique())
    T = len(times)
    print("Total unique times:", T)

    full_index = pd.MultiIndex.from_product(
        [times, stations["StationID"].values],
        names=["Datetime", "StationID"]
    )

    meteo_cols = [
        "DewPointTemperature",
        "StationLevelPressure",
        "SP",
        "Humidity",
        "Rainfall",
        "DR_sin",
        "DR_cos",
    ]

    df2 = df.set_index(["Datetime", "StationID"])[meteo_cols].sort_index()
    df_full = df2.reindex(full_index)
    X_raw = df_full.values.reshape(T, N, len(meteo_cols)).astype(np.float32)

    M_in = (~np.isnan(X_raw)).astype(np.float32)

    rain_idx = meteo_cols.index("Rainfall")
    Y_rain = X_raw[..., rain_idx]
    M_y    = M_in[..., rain_idx]

    # time-of-day / month features
    dt_index = pd.DatetimeIndex(times)
    hour = dt_index.hour.astype(np.float32)
    month = dt_index.month.astype(np.float32)

    hour_sin = np.sin(2*np.pi*(hour/24.0))
    hour_cos = np.cos(2*np.pi*(hour/24.0))
    month_sin = np.sin(2*np.pi*((month-1)/12.0))
    month_cos = np.cos(2*np.pi*((month-1)/12.0))

    time_feats = np.stack([hour_sin, hour_cos, month_sin, month_cos], axis=-1).astype(np.float32)
    time_feats = np.repeat(time_feats[:, None, :], N, axis=1)

    # regime ids (season)
    season_by_time = (
        df.groupby("Datetime")["Season"]
          .agg(lambda s: s.mode().iloc[0] if len(s.mode()) else s.iloc[0])
          .reindex(times)
          .astype(str)
          .values
    )
    unique_seasons = sorted(pd.unique(season_by_time))
    season_to_id = {s: i for i, s in enumerate(unique_seasons)}
    season_ids = np.array([season_to_id[s] for s in season_by_time], dtype=np.int64)

    return {
        "stations": stations,
        "times": times,
        "X_raw": X_raw,
        "M_in": M_in,
        "Y_rain": Y_rain,
        "M_y": M_y,
        "meteo_cols": meteo_cols,
        "time_feats": time_feats,
        "season_ids": season_ids,
        "unique_seasons": unique_seasons,
    }

# ============================================================
# 3) Dataset (same structure as NN baselines)
# ============================================================

class BanglaRainDataset(Dataset):
    def __init__(
        self,
        X_scaled, M_in, time_feats,
        Y_rain, M_y,
        Acc24, MaskAcc24,
        season_ids,
        thr3h, thrAcc24,
        t_start, t_end,
        T_in=16, H_out=8,
        peak_min_obs=None,
    ):
        self.X_scaled = X_scaled
        self.M_in = M_in
        self.time_feats = time_feats
        self.Y_rain = Y_rain
        self.M_y = M_y
        self.Acc24 = Acc24
        self.MaskAcc24 = MaskAcc24
        self.season_ids = season_ids
        self.thr3h = thr3h
        self.thrAcc24 = thrAcc24
        self.T_in = T_in
        self.H_out = H_out
        self.peak_min_obs = peak_min_obs if peak_min_obs is not None else (H_out // 2)

        self.indices = []
        for t in range(t_start + (T_in - 1), t_end - H_out):
            self.indices.append(t)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        t = self.indices[idx]

        x  = self.X_scaled[t-self.T_in+1:t+1]     # [T_in,N,F]
        m  = self.M_in[t-self.T_in+1:t+1]         # [T_in,N,F]
        tf = self.time_feats[t-self.T_in+1:t+1]   # [T_in,N,4]

        x = np.nan_to_num(x, nan=0.0).astype(np.float32)
        x_all = np.concatenate([x, m.astype(np.float32), tf], axis=-1)  # [T_in,N,F+F+4]

        y  = self.Y_rain[t+1:t+1+self.H_out]       # [H_out,N]
        my = self.M_y[t+1:t+1+self.H_out].astype(np.float32)
        y_log = np.log1p(np.nan_to_num(y, nan=0.0)).astype(np.float32)

        # flash at t+1
        y_next = self.Y_rain[t+1]
        m_next = self.M_y[t+1].astype(np.float32)
        flash  = ((y_next >= self.thr3h) & (m_next > 0.5)).astype(np.float32)
        mflash = m_next.copy()

        # peak24
        y_win = self.Y_rain[t+1:t+1+self.H_out]
        m_win = self.M_y[t+1:t+1+self.H_out].astype(np.float32)
        mpeak = (m_win.sum(axis=0) >= self.peak_min_obs).astype(np.float32)
        y_peak = np.nanmax(np.where(m_win > 0.5, y_win, np.nan), axis=0)
        peak24 = ((y_peak >= self.thr3h) & (mpeak > 0.5)).astype(np.float32)

        # acc24
        acc  = self.Acc24[t]
        macc = self.MaskAcc24[t].astype(np.float32)
        acc24 = ((acc >= self.thrAcc24) & (macc > 0.5)).astype(np.float32)

        regime_id = int(self.season_ids[t])

        return (
            torch.from_numpy(x_all),                         # x
            torch.tensor(regime_id, dtype=torch.long),       # regime
            torch.from_numpy(y_log),                         # y_log
            torch.from_numpy(my),                            # my
            torch.from_numpy(flash), torch.from_numpy(mflash),
            torch.from_numpy(peak24), torch.from_numpy(mpeak),
            torch.from_numpy(acc24),  torch.from_numpy(macc),
        )

# ============================================================
# 4) Build data objects + loaders
# ============================================================

def build_data_objects(df_rain, T_in=16, H_out=8, train_frac=0.7, val_frac=0.15):
    prep = prepare_full_grid(df_rain)
    T = len(prep["times"])
    train_end, val_end = make_splits(T, train_frac=train_frac, val_frac=val_frac)

    X_scaled, scaler = scale_inputs_train_only(prep["X_raw"], train_end)

    thr3h = compute_thresholds_train(prep["Y_rain"], prep["M_y"], train_end, q=0.95)

    Acc24, MaskAcc24 = compute_acc24(prep["Y_rain"], prep["M_y"], H_out=H_out)
    thrAcc24 = compute_thresholds_train(Acc24, MaskAcc24, train_end, q=0.95)

    ds_train = BanglaRainDataset(
        X_scaled, prep["M_in"], prep["time_feats"],
        prep["Y_rain"], prep["M_y"],
        Acc24, MaskAcc24,
        prep["season_ids"],
        thr3h, thrAcc24,
        t_start=0, t_end=train_end,
        T_in=T_in, H_out=H_out
    )
    ds_val = BanglaRainDataset(
        X_scaled, prep["M_in"], prep["time_feats"],
        prep["Y_rain"], prep["M_y"],
        Acc24, MaskAcc24,
        prep["season_ids"],
        thr3h, thrAcc24,
        t_start=train_end, t_end=val_end,
        T_in=T_in, H_out=H_out
    )
    ds_test = BanglaRainDataset(
        X_scaled, prep["M_in"], prep["time_feats"],
        prep["Y_rain"], prep["M_y"],
        Acc24, MaskAcc24,
        prep["season_ids"],
        thr3h, thrAcc24,
        t_start=val_end, t_end=T,
        T_in=T_in, H_out=H_out
    )

    return prep, scaler, (thr3h, thrAcc24), (Acc24, MaskAcc24), (train_end, val_end, T), (ds_train, ds_val, ds_test)

def make_loaders(ds_train, ds_val, ds_test, batch_size=256):
    train_loader = DataLoader(ds_train, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader   = DataLoader(ds_val,   batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(ds_test,  batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader

# ============================================================
# 5) Flatten loader -> tabular data for LightGBM
# ============================================================

def build_tabular_from_loader(loader):
    """
    Build two tabular datasets:
      - Regression: per (time, lead, station) rainfall in mm.
      - Classification: per (time, station) flash/peak/acc.
    """
    # Regression
    X_reg_list = []
    y_reg_list = []
    station_reg_list = []
    lead_reg_list = []
    mask_reg_list = []

    # Classification
    X_clf_list = []
    station_clf_list = []
    flash_y_list, flash_m_list = [], []
    peak_y_list,  peak_m_list  = [], []
    acc_y_list,   acc_m_list   = [], []

    for (x, reg, y_log, my, flash, mflash, peak, mpeak, acc, macc) in loader:
        x_np = x[:, -1, :, :].numpy()  # [B,N,F_all] last step
        y_mm = torch.expm1(y_log).clamp_min(0.0).numpy()  # [B,H,N]
        my_np = my.numpy()

        flash_np = flash.numpy()
        mflash_np = mflash.numpy()
        peak_np  = peak.numpy()
        mpeak_np = mpeak.numpy()
        acc_np   = acc.numpy()
        macc_np  = macc.numpy()

        B, H, N = y_mm.shape

        # --- Regression rows: (time, lead, station) ---
        for b in range(B):
            for h in range(H):
                mask_h = my_np[b, h, :] > 0.5
                if not mask_h.any():
                    continue
                X_bh = x_np[b]  # [N,F]
                y_bh = y_mm[b, h, :]
                for j in range(N):
                    if mask_h[j]:
                        X_reg_list.append(X_bh[j])
                        y_reg_list.append(y_bh[j])
                        station_reg_list.append(j)
                        lead_reg_list.append(h)
                        mask_reg_list.append(1.0)

        # --- Classification rows: (time, station) ---
        for b in range(B):
            for j in range(N):
                if mflash_np[b, j] > 0.5:
                    X_clf_list.append(x_np[b, j])
                    station_clf_list.append(j)
                    flash_y_list.append(flash_np[b, j])
                    peak_y_list.append(peak_np[b, j])
                    acc_y_list.append(acc_np[b, j])
                    flash_m_list.append(mflash_np[b, j])
                    peak_m_list.append(mpeak_np[b, j])
                    acc_m_list.append(macc_np[b, j])

    X_reg = np.array(X_reg_list, dtype=np.float32)
    y_reg = np.array(y_reg_list, dtype=np.float32)
    station_reg = np.array(station_reg_list, dtype=np.int64)
    lead_reg = np.array(lead_reg_list, dtype=np.int64)
    mask_reg = np.array(mask_reg_list, dtype=np.float32)

    X_clf = np.array(X_clf_list, dtype=np.float32)
    station_clf = np.array(station_clf_list, dtype=np.int64)
    flash_y = np.array(flash_y_list, dtype=np.float32)
    flash_m = np.array(flash_m_list, dtype=np.float32)
    peak_y  = np.array(peak_y_list, dtype=np.float32)
    peak_m  = np.array(peak_m_list, dtype=np.float32)
    acc_y   = np.array(acc_y_list, dtype=np.float32)
    acc_m   = np.array(acc_m_list, dtype=np.float32)

    return {
        "X_reg": X_reg,
        "y_reg": y_reg,
        "station_reg": station_reg,
        "lead_reg": lead_reg,
        "mask_reg": mask_reg,

        "X_clf": X_clf,
        "station_clf": station_clf,
        "flash_y": flash_y,
        "flash_m": flash_m,
        "peak_y": peak_y,
        "peak_m": peak_m,
        "acc_y": acc_y,
        "acc_m": acc_m,
    }

def merge_tabular(tab_list):
    """Merge several tabular dicts (TRAIN + VAL) into one."""
    out = {}
    keys = [
        "X_reg", "y_reg", "station_reg", "lead_reg", "mask_reg",
        "X_clf", "station_clf",
        "flash_y", "flash_m",
        "peak_y",  "peak_m",
        "acc_y",   "acc_m",
    ]
    for k in keys:
        arrays = [t[k] for t in tab_list if t[k] is not None and len(t[k]) > 0]
        out[k] = np.concatenate(arrays, axis=0) if arrays else np.array([], dtype=np.float32)
    return out

# ============================================================
# 6) LightGBM training helpers
# ============================================================

def train_station_regressors(X_reg, y_reg, station_reg, params_base, quantiles, num_stations):
    lgbm_regressors = {}
    for j in range(num_stations):
        mask_j = (station_reg == j)
        if not np.any(mask_j):
            continue
        X_j = X_reg[mask_j]
        y_j = y_reg[mask_j]
        if X_j.shape[0] < 50:
            continue

        for q in quantiles:
            params = params_base.copy()
            params.update({
                "objective": "quantile",
                "alpha": q,
                "metric": "quantile",
            })
            dtrain = lgb.Dataset(X_j, label=y_j)
            model = lgb.train(
                params,
                dtrain,
                num_boost_round=params["n_estimators"]
            )
            lgbm_regressors[(j, q)] = model
    return lgbm_regressors

def train_station_classifiers(tab, num_stations, params_base):
    X_clf = tab["X_clf"]
    station_clf = tab["station_clf"]

    def train_one_target(y, m):
        models = {}
        for j in range(num_stations):
            mask_station = (station_clf == j)
            if not np.any(mask_station):
                continue

            X_j = X_clf[mask_station]
            y_j = y[mask_station]
            m_j = m[mask_station]

            # require valid label
            valid = (m_j > 0.5)
            X_j = X_j[valid]
            y_j = y_j[valid]

            if X_j.shape[0] < 20:
                # dummy model predicting all zeros
                X_dummy = np.zeros((50, X_clf.shape[1]), dtype=np.float32)
                y_dummy = np.zeros(50, dtype=np.float32)
                params = params_base.copy()
                params.update({
                    "objective": "binary",
                    "metric": "binary_logloss",
                })
                dtrain = lgb.Dataset(X_dummy, label=y_dummy)
                model = lgb.train(
                    params,
                    dtrain,
                    num_boost_round=10
                )
                models[j] = model
                continue

            params = params_base.copy()
            params.update({
                "objective": "binary",
                "metric": "binary_logloss",
            })
            dtrain = lgb.Dataset(X_j, label=y_j)
            model = lgb.train(
                params,
                dtrain,
                num_boost_round=params["n_estimators"]
            )
            models[j] = model
        return models

    lgbm_flash = train_one_target(tab["flash_y"], tab["flash_m"])
    lgbm_peak  = train_one_target(tab["peak_y"],  tab["peak_m"])
    lgbm_acc   = train_one_target(tab["acc_y"],   tab["acc_m"])

    return lgbm_flash, lgbm_peak, lgbm_acc

# ============================================================
# 7) Load best hyperparameters from tuning
# ============================================================

def load_best_hparams(best_trial_path):
    with open(best_trial_path, "r") as f:
        best = json.load(f)
    # hyperparams used in tuning
    hp = {
        "learning_rate":    best["learning_rate"],
        "n_estimators":     best["n_estimators"],
        "num_leaves":       best["num_leaves"],
        "min_data_in_leaf": best["min_data_in_leaf"],
        "feature_fraction": best["feature_fraction"],
        "bagging_fraction": best["bagging_fraction"],
        "bagging_freq":     best["bagging_freq"],
        "lambda_l2":        best["lambda_l2"],
    }
    return hp, best

# ============================================================
# 8) Final training function (TRAIN + VAL, no test)
# ============================================================

def final_train_lightgbm_stationwise(
    df_rain,
    T_in=16,
    H_out=8,
    train_frac=0.7,
    val_frac=0.15,
    tuning_dir=TUNING_SAVE_DIR,
    final_dir=FINAL_SAVE_DIR,
):
    # 1) Load best hyperparameters from tuning
    best_trial_path = os.path.join(tuning_dir, "best_trial.json")
    print("\nLoading best hyperparameters from:", best_trial_path)
    hp, best_trial_full = load_best_hparams(best_trial_path)
    print("Best tuning trial summary:", best_trial_full)

    # 2) Build data objects (same preprocessing as tuning)
    prep, scaler, (thr3h, thrAcc24), (Acc24, MaskAcc24), splits, datasets = build_data_objects(
        df_rain,
        T_in=T_in,
        H_out=H_out,
        train_frac=train_frac,
        val_frac=val_frac,
    )
    train_end, val_end, T_total = splits
    ds_train, ds_val, ds_test = datasets

    print("\nData split indices:")
    print("  Train: 0 →", train_end)
    print("  Val:   ", train_end, "→", val_end)
    print("  Test:  ", val_end, "→", T_total)

    train_loader, val_loader, _ = make_loaders(
        ds_train, ds_val, ds_test, batch_size=256
    )

    N_stations = len(prep["stations"])
    print("N_stations:", N_stations)

    # 3) Build tabular TRAIN + VAL data
    print("\nBuilding tabular TRAIN data...")
    tab_train = build_tabular_from_loader(train_loader)
    print("  TRAIN: X_reg:", tab_train["X_reg"].shape,
          "| y_reg:", tab_train["y_reg"].shape)
    print("         X_clf:", tab_train["X_clf"].shape)

    print("\nBuilding tabular VAL data (to include in final training)...")
    tab_val = build_tabular_from_loader(val_loader)
    print("  VAL:   X_reg:", tab_val["X_reg"].shape,
          "| y_reg:", tab_val["y_reg"].shape)
    print("         X_clf:", tab_val["X_clf"].shape)

    print("\nMerging TRAIN + VAL tabular datasets...")
    tab_full = merge_tabular([tab_train, tab_val])
    print("  FULL:  X_reg:", tab_full["X_reg"].shape,
          "| y_reg:", tab_full["y_reg"].shape)
    print("         X_clf:", tab_full["X_clf"].shape)

    # 4) Build base LightGBM params from best hyperparams
    params_base = {
        "boosting_type":      "gbdt",
        "num_leaves":         int(hp["num_leaves"]),
        "min_data_in_leaf":   int(hp["min_data_in_leaf"]),
        "feature_fraction":   float(hp["feature_fraction"]),
        "bagging_fraction":   float(hp["bagging_fraction"]),
        "bagging_freq":       int(hp["bagging_freq"]),
        "lambda_l2":          float(hp["lambda_l2"]),
        "learning_rate":      float(hp["learning_rate"]),
        "verbosity":          -1,
        "seed":               42,
        "feature_fraction_seed": 42,
        "bagging_seed":       42,
        "deterministic":      True,
        "force_row_wise":     True,
        "num_threads":        -1,
        "n_estimators":       int(hp["n_estimators"]),
    }

    quantiles = (0.1, 0.5, 0.9)

    # 5) Train station-wise regressors / classifiers on FULL (TRAIN+VAL) tabular
    print("\n[FINAL LightGBM] Training station-wise quantile regressors...")
    lgbm_reg = train_station_regressors(
        tab_full["X_reg"],
        tab_full["y_reg"],
        tab_full["station_reg"],
        params_base,
        quantiles=quantiles,
        num_stations=N_stations,
    )

    print("\n[FINAL LightGBM] Training station-wise classifiers (flash/peak/acc)...")
    lgbm_flash, lgbm_peak, lgbm_acc = train_station_classifiers(
        tab_full,
        num_stations=N_stations,
        params_base=params_base,
    )

    # 6) Save final models + preprocessing artifacts
    final_artifacts = {
        "hyperparams_best_trial": best_trial_full,
        "params_base": params_base,
        "quantiles": quantiles,
        "lgbm_reg": lgbm_reg,
        "lgbm_flash": lgbm_flash,
        "lgbm_peak": lgbm_peak,
        "lgbm_acc": lgbm_acc,
        "scaler_mean": getattr(scaler, "mean_", None),
        "scaler_std": getattr(scaler, "std_", None),
        "thr3h": thr3h,
        "thrAcc24": thrAcc24,
        "stations": prep["stations"],
        "times": prep["times"],
        "T_in": int(T_in),
        "H_out": int(H_out),
        "train_frac": float(train_frac),
        "val_frac": float(val_frac),
    }

    out_path = os.path.join(final_dir, "final_lightgbm_models.pkl")
    joblib.dump(final_artifacts, out_path)
    print("\n[FINAL LightGBM] Training complete.")
    print("Final models + preprocessing saved to:")
    print("  ", out_path)

    return {
        "models_path": out_path,
        "n_stations": N_stations,
        "train_end": train_end,
        "val_end": val_end,
        "T_total": T_total,
    }

# ============================================================
# 9) RUN FINAL TRAINING (CALL THIS AFTER df_rain IS LOADED)
# ============================================================

results = final_train_lightgbm_stationwise(
    df_rain=df_rain,
    T_in=16,
    H_out=8,
    train_frac=0.7,
    val_frac=0.15,
    tuning_dir=TUNING_SAVE_DIR,
    final_dir=FINAL_SAVE_DIR,
)

print("\nFinal LightGBM training summary:")
print(results)


Using device: cpu
Tuning directory : C:\Users\pc\Documents\Rainfall\Baselines\LightGBM\Tuning_v1
Final save dir   : C:\Users\pc\Documents\Rainfall\Baselines\LightGBM\Final_Run\v1

Loading best hyperparameters from: C:\Users\pc\Documents\Rainfall\Baselines\LightGBM\Tuning_v1\best_trial.json
Best tuning trial summary: {'trial': 2, 'val_CRPS_log': 0.13871812798283323, 'learning_rate': 0.013728250383906291, 'n_estimators': 763, 'num_leaves': 40, 'min_data_in_leaf': 150, 'feature_fraction': 0.5102922471479012, 'bagging_fraction': 0.9849549260809971, 'bagging_freq': 4, 'lambda_l2': 42.78743516759823}
Total stations: 34
Total unique times: 61360

Data split indices:
  Train: 0 → 42952
  Val:    42952 → 52156
  Test:   52156 → 61360
N_stations: 34

Building tabular TRAIN data...


C:\Users\pc\AppData\Local\Temp\ipykernel_9604\3569106490.py:338: RuntimeWarning: All-NaN slice encountered
  y_peak = np.nanmax(np.where(m_win > 0.5, y_win, np.nan), axis=0)


  TRAIN: X_reg: (11538022, 18) | y_reg: (11538022,)
         X_clf: (1442245, 18)

Building tabular VAL data (to include in final training)...
  VAL:   X_reg: (2494848, 18) | y_reg: (2494848,)
         X_clf: (311856, 18)

Merging TRAIN + VAL tabular datasets...
  FULL:  X_reg: (14032870, 18) | y_reg: (14032870,)
         X_clf: (1754101, 18)

[FINAL LightGBM] Training station-wise quantile regressors...

[FINAL LightGBM] Training station-wise classifiers (flash/peak/acc)...

[FINAL LightGBM] Training complete.
Final models + preprocessing saved to:
   C:\Users\pc\Documents\Rainfall\Baselines\LightGBM\Final_Run\v1\final_lightgbm_models.pkl

Final LightGBM training summary:
{'models_path': 'C:\\Users\\pc\\Documents\\Rainfall\\Baselines\\LightGBM\\Final_Run\\v1\\final_lightgbm_models.pkl', 'n_stations': 34, 'train_end': 42952, 'val_end': 52156, 'T_total': 61360}


# Testing

In [ ]:
# ============================================================
# STATION-WISE LIGHTGBM BASELINE - TESTING PIPELINE
# ============================================================
#
# - Assumes df_rain (DataFrame) is ALREADY LOADED in memory.
# - Assumes final training produced:
#       FINAL_SAVE_DIR/final_lightgbm_models.pkl
# - This script:
#       * Rebuilds preprocessing and splits (same as training)
#       * Uses SAVED scaler_mean/std, thr3h, thrAcc24, hyperparams
#       * Wraps the saved station-wise LightGBM models
#       * Evaluates on TEST split only (no retraining)
#       * Prints same metrics as other baselines:
#           - CRPS_log, CRPS_mm
#           - Brier_flash / Brier_peak / Brier_acc
#           - Event metrics (PR-AUC/ROC-AUC, POD/FAR/CSI)
#           - MAE/RMSE by lead
# ============================================================

import os, random, math, json
import numpy as np
import pandas as pd
import lightgbm as lgb
import joblib

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import average_precision_score, roc_auc_score

# ------------------------------------------------------------
# 0) Reproducibility + Device
# ------------------------------------------------------------
def set_seed(seed=42, deterministic=False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42, deterministic=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ------------------------------------------------------------
# 0.5) Paths (EDIT THESE IF NEEDED)
# ------------------------------------------------------------
FINAL_SAVE_DIR = r"C:\Users\pc\Documents\Rainfall\Baselines\LightGBM\Final_Run\v1"
os.makedirs(FINAL_SAVE_DIR, exist_ok=True)

FINAL_MODELS_PATH = os.path.join(FINAL_SAVE_DIR, "final_lightgbm_models.pkl")

print("Final models path :", FINAL_MODELS_PATH)

# ============================================================
# 1) Core utilities (must match training)
# ============================================================

class NaNIgnoringStandardScaler:
    def __init__(self, eps=1e-8):
        self.eps = eps
        self.mean_ = None
        self.std_ = None

    def fit(self, X):
        self.mean_ = np.nanmean(X, axis=0)
        self.std_  = np.nanstd(X, axis=0)
        self.std_  = np.where(self.std_ < self.eps, 1.0, self.std_)
        return self

    def transform(self, X):
        std_safe = np.where((self.std_ is not None) & (self.std_ < self.eps), 1.0, self.std_)
        return (X - self.mean_) / std_safe

def pinball_loss(y_hat_q, y_true, q, mask=None):
    e = y_true - y_hat_q
    loss = torch.maximum((q - 1.0) * e, q * e)
    if mask is not None:
        loss = loss * mask
        denom = mask.sum().clamp_min(1.0)
        return loss.sum() / denom
    return loss.mean()

def crps_from_quantiles(y_hat, y_true, quantiles, mask=None):
    losses = []
    for k, q in enumerate(quantiles):
        losses.append(pinball_loss(y_hat[..., k], y_true, q, mask=mask))
    return 2.0 * torch.stack(losses).mean()

def brier_score(probs, targets, mask=None):
    loss = (probs - targets) ** 2
    if mask is not None:
        loss = loss * mask
        denom = mask.sum().clamp_min(1.0)
        return loss.sum() / denom
    return loss.mean()

def safe_div(a, b, eps=1e-12):
    return a / (b + eps)

def event_metrics_binary(probs, y_true, mask, thresholds=(0.1, 0.2, 0.3, 0.5)):
    valid = mask > 0.5
    if valid.sum() < 50:
        return {
            "n_valid": int(valid.sum()),
            "pr_auc": np.nan,
            "roc_auc": np.nan,
            "by_thr": {thr: {"POD": np.nan, "FAR": np.nan, "CSI": np.nan} for thr in thresholds},
        }

    p = probs[valid]
    y = y_true[valid]

    pr_auc = average_precision_score(y, p) if (y.max() > 0 and y.min() < 1) else np.nan
    try:
        roc_auc = roc_auc_score(y, p) if (y.max() > 0 and y.min() < 1) else np.nan
    except Exception:
        roc_auc = np.nan

    by_thr = {}
    for thr in thresholds:
        yhat = (p >= thr).astype(np.float32)
        TP = ((yhat == 1) & (y == 1)).sum()
        FP = ((yhat == 1) & (y == 0)).sum()
        FN = ((yhat == 0) & (y == 1)).sum()

        POD = safe_div(TP, TP + FN)
        FAR = safe_div(FP, TP + FP)
        CSI = safe_div(TP, TP + FP + FN)

        by_thr[thr] = {"POD": float(POD), "FAR": float(FAR), "CSI": float(CSI)}

    return {
        "n_valid": int(valid.sum()),
        "pr_auc": float(pr_auc) if pr_auc == pr_auc else np.nan,
        "roc_auc": float(roc_auc) if roc_auc == roc_auc else np.nan,
        "by_thr": by_thr,
    }

def make_splits(T, train_frac=0.7, val_frac=0.15):
    train_end = int(T * train_frac)
    val_end   = int(T * (train_frac + val_frac))
    return train_end, val_end

def compute_acc24(Y_rain, M_y, H_out=8):
    T, N = Y_rain.shape
    Acc  = np.full((T, N), np.nan, dtype=np.float32)
    Mask = np.zeros((T, N), dtype=np.float32)
    for t in range(T - H_out):
        window = Y_rain[t+1:t+1+H_out]
        wmask  = M_y[t+1:t+1+H_out]
        ok = (wmask.sum(axis=0) == H_out)
        Mask[t, ok] = 1.0
        Acc[t, ok]  = window[:, ok].sum(axis=0)
    return Acc, Mask

# ============================================================
# 2) Preprocessing: full grid, station/time features
# ============================================================

def prepare_full_grid(df_rain):
    df = df_rain.copy()
    df["Datetime"] = pd.to_datetime(df["Datetime"])
    df = df.sort_values(["Datetime", "StationID"]).reset_index(drop=True)

    # DR -> sin/cos
    dr = df["DR"].astype(np.float32).to_numpy()
    dr_rad = np.deg2rad(dr)
    df["DR_sin"] = np.sin(dr_rad)
    df["DR_cos"] = np.cos(dr_rad)

    stations = (
        df[["StationID", "StationName", "Latitude", "Longitude"]]
        .drop_duplicates()
        .sort_values("StationID")
        .reset_index(drop=True)
    )
    stations["node_index"] = np.arange(len(stations))
    N = len(stations)
    print("Total stations:", N)

    times = np.sort(df["Datetime"].unique())
    T = len(times)
    print("Total unique times:", T)

    full_index = pd.MultiIndex.from_product(
        [times, stations["StationID"].values],
        names=["Datetime", "StationID"]
    )

    meteo_cols = [
        "DewPointTemperature",
        "StationLevelPressure",
        "SP",
        "Humidity",
        "Rainfall",
        "DR_sin",
        "DR_cos",
    ]

    df2 = df.set_index(["Datetime", "StationID"])[meteo_cols].sort_index()
    df_full = df2.reindex(full_index)
    X_raw = df_full.values.reshape(T, N, len(meteo_cols)).astype(np.float32)

    M_in = (~np.isnan(X_raw)).astype(np.float32)

    rain_idx = meteo_cols.index("Rainfall")
    Y_rain = X_raw[..., rain_idx]
    M_y    = M_in[..., rain_idx]

    # time-of-day / month features
    dt_index = pd.DatetimeIndex(times)
    hour = dt_index.hour.astype(np.float32)
    month = dt_index.month.astype(np.float32)

    hour_sin = np.sin(2*np.pi*(hour/24.0))
    hour_cos = np.cos(2*np.pi*(hour/24.0))
    month_sin = np.sin(2*np.pi*((month-1)/12.0))
    month_cos = np.cos(2*np.pi*((month-1)/12.0))

    time_feats = np.stack([hour_sin, hour_cos, month_sin, month_cos], axis=-1).astype(np.float32)
    time_feats = np.repeat(time_feats[:, None, :], N, axis=1)

    # regime ids (season)
    season_by_time = (
        df.groupby("Datetime")["Season"]
          .agg(lambda s: s.mode().iloc[0] if len(s.mode()) else s.iloc[0])
          .reindex(times)
          .astype(str)
          .values
    )
    unique_seasons = sorted(pd.unique(season_by_time))
    season_to_id = {s: i for i, s in enumerate(unique_seasons)}
    season_ids = np.array([season_to_id[s] for s in season_by_time], dtype=np.int64)

    return {
        "stations": stations,
        "times": times,
        "X_raw": X_raw,
        "M_in": M_in,
        "Y_rain": Y_rain,
        "M_y": M_y,
        "meteo_cols": meteo_cols,
        "time_feats": time_feats,
        "season_ids": season_ids,
        "unique_seasons": unique_seasons,
    }

# ============================================================
# 3) Dataset (same structure as NN baselines)
# ============================================================

class BanglaRainDataset(Dataset):
    def __init__(
        self,
        X_scaled, M_in, time_feats,
        Y_rain, M_y,
        Acc24, MaskAcc24,
        season_ids,
        thr3h, thrAcc24,
        t_start, t_end,
        T_in=16, H_out=8,
        peak_min_obs=None,
    ):
        self.X_scaled = X_scaled
        self.M_in = M_in
        self.time_feats = time_feats
        self.Y_rain = Y_rain
        self.M_y = M_y
        self.Acc24 = Acc24
        self.MaskAcc24 = MaskAcc24
        self.season_ids = season_ids
        self.thr3h = thr3h
        self.thrAcc24 = thrAcc24
        self.T_in = T_in
        self.H_out = H_out
        self.peak_min_obs = peak_min_obs if peak_min_obs is not None else (H_out // 2)

        self.indices = []
        for t in range(t_start + (T_in - 1), t_end - H_out):
            self.indices.append(t)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        t = self.indices[idx]

        x  = self.X_scaled[t-self.T_in+1:t+1]     # [T_in,N,F]
        m  = self.M_in[t-self.T_in+1:t+1]         # [T_in,N,F]
        tf = self.time_feats[t-self.T_in+1:t+1]   # [T_in,N,4]

        x = np.nan_to_num(x, nan=0.0).astype(np.float32)
        x_all = np.concatenate([x, m.astype(np.float32), tf], axis=-1)  # [T_in,N,F+F+4]

        y  = self.Y_rain[t+1:t+1+self.H_out]       # [H_out,N]
        my = self.M_y[t+1:t+1+self.H_out].astype(np.float32)
        y_log = np.log1p(np.nan_to_num(y, nan=0.0)).astype(np.float32)

        # flash at t+1
        y_next = self.Y_rain[t+1]
        m_next = self.M_y[t+1].astype(np.float32)
        flash  = ((y_next >= self.thr3h) & (m_next > 0.5)).astype(np.float32)
        mflash = m_next.copy()

        # peak24
        y_win = self.Y_rain[t+1:t+1+self.H_out]
        m_win = self.M_y[t+1:t+1+self.H_out].astype(np.float32)
        mpeak = (m_win.sum(axis=0) >= self.peak_min_obs).astype(np.float32)
        y_peak = np.nanmax(np.where(m_win > 0.5, y_win, np.nan), axis=0)
        peak24 = ((y_peak >= self.thr3h) & (mpeak > 0.5)).astype(np.float32)

        # acc24
        acc  = self.Acc24[t]
        macc = self.MaskAcc24[t].astype(np.float32)
        acc24 = ((acc >= self.thrAcc24) & (macc > 0.5)).astype(np.float32)

        regime_id = int(self.season_ids[t])

        return (
            torch.from_numpy(x_all),                         # x
            torch.tensor(regime_id, dtype=torch.long),       # regime
            torch.from_numpy(y_log),                         # y_log
            torch.from_numpy(my),                            # my
            torch.from_numpy(flash), torch.from_numpy(mflash),
            torch.from_numpy(peak24), torch.from_numpy(mpeak),
            torch.from_numpy(acc24),  torch.from_numpy(macc),
        )

def make_loaders(ds_test, batch_size=256):
    test_loader  = DataLoader(ds_test, batch_size=batch_size, shuffle=False)
    return test_loader

# ============================================================
# 4) Station-wise LightGBM model wrapper (same as tuning)
# ============================================================

class StationWiseLGBMBaseline(nn.Module):
    """
    LightGBM station-wise baseline:
      - One quantile regressor per (station, quantile).
      - One binary classifier per station for flash / peak / acc.
    Wrapped as nn.Module so `evaluate()` works unchanged.
    """
    def __init__(self,
                 lgbm_regressors,
                 lgbm_flash,
                 lgbm_peak,
                 lgbm_acc,
                 num_stations,
                 H_out,
                 quantiles):
        super().__init__()
        self.lgbm_regressors = lgbm_regressors
        self.lgbm_flash = lgbm_flash
        self.lgbm_peak = lgbm_peak
        self.lgbm_acc = lgbm_acc
        self.N = num_stations
        self.H_out = H_out
        self.quantiles = list(quantiles)
        self.K = len(self.quantiles)

    def forward(self, x, regime_id):
        B, T, N, F = x.shape
        assert N == self.N

        # last time step features
        x_last = x[:, -1, :, :].detach().cpu().numpy()  # [B,N,F]

        q_mm = np.zeros((B, self.H_out, N, self.K), dtype=np.float32)
        flash_prob = np.zeros((B, N), dtype=np.float32)
        peak_prob  = np.zeros((B, N), dtype=np.float32)
        acc_prob   = np.zeros((B, N), dtype=np.float32)

        for j in range(N):
            X_j = x_last[:, j, :]  # [B,F]

            # Quantile regressors
            for k, q in enumerate(self.quantiles):
                model = self.lgbm_regressors.get((j, q), None)
                if model is None:
                    yhat = np.zeros(B, dtype=np.float32)
                else:
                    yhat = model.predict(X_j)
                q_mm[:, :, j, k] = yhat[:, None]  # same for all H_out

            # Binary classifiers: predict probabilities
            m_flash = self.lgbm_flash.get(j, None)
            if m_flash is None:
                flash_prob[:, j] = 0.0
            else:
                p = m_flash.predict(X_j)
                flash_prob[:, j] = p if p.ndim == 1 else p[:, 1]

            m_peak = self.lgbm_peak.get(j, None)
            if m_peak is None:
                peak_prob[:, j] = 0.0
            else:
                p = m_peak.predict(X_j)
                peak_prob[:, j] = p if p.ndim == 1 else p[:, 1]

            m_acc = self.lgbm_acc.get(j, None)
            if m_acc is None:
                acc_prob[:, j] = 0.0
            else:
                p = m_acc.predict(X_j)
                acc_prob[:, j] = p if p.ndim == 1 else p[:, 1]

        q_mm_tensor = torch.from_numpy(q_mm).to(device)
        q_log = torch.log1p(q_mm_tensor).clamp_min(0.0)

        def prob_to_logits(p):
            p = torch.clamp(p, 1e-6, 1.0 - 1e-6)
            return torch.log(p / (1.0 - p))

        flash_logits = prob_to_logits(torch.from_numpy(flash_prob).to(device))
        peak_logits  = prob_to_logits(torch.from_numpy(peak_prob).to(device))
        acc_logits   = prob_to_logits(torch.from_numpy(acc_prob).to(device))

        return q_log, flash_logits, peak_logits, acc_logits

# ============================================================
# 5) Evaluation (same as other baselines)
# ============================================================

@torch.no_grad()
def evaluate(model, loader, quantiles, thresholds=(0.1, 0.2, 0.3, 0.5)):
    model.eval()

    total_crps_log = 0.0
    total_crps_mm  = 0.0
    total_brier_flash = 0.0
    total_brier_peak  = 0.0
    total_brier_acc   = 0.0
    nb = 0

    H_out = None
    sum_abs = None
    sum_sq  = None
    count   = None

    qs = np.array(list(quantiles), dtype=np.float32)
    k50 = int(np.argmin(np.abs(qs - 0.5)))

    flash_p_list, flash_y_list, flash_m_list = [], [], []
    peak_p_list,  peak_y_list,  peak_m_list  = [], [], []
    acc_p_list,   acc_y_list,   acc_m_list   = [], [], []

    for (x, reg, y_log, my, flash, mflash, peak, mpeak, acc, macc) in loader:
        x = x.to(device)
        reg = reg.to(device)
        y_log = y_log.to(device)
        my = my.to(device)
        flash = flash.to(device); mflash = mflash.to(device)
        peak  = peak.to(device);  mpeak  = mpeak.to(device)
        acc   = acc.to(device);   macc   = macc.to(device)

        q_hat, flash_logits, peak_logits, acc_logits = model(x, reg)

        # CRPS (log space)
        total_crps_log += crps_from_quantiles(q_hat, y_log, quantiles, mask=my).item()

        # Convert to mm and compute CRPS
        q_mm = torch.expm1(q_hat).clamp_min(0.0)
        y_mm = torch.expm1(y_log).clamp_min(0.0)
        total_crps_mm  += crps_from_quantiles(q_mm, y_mm, quantiles, mask=my).item()

        # Risk Brier
        p_flash = torch.sigmoid(flash_logits)
        p_peak  = torch.sigmoid(peak_logits)
        p_acc   = torch.sigmoid(acc_logits)

        total_brier_flash += brier_score(p_flash, flash, mask=mflash).item()
        total_brier_peak  += brier_score(p_peak,  peak,  mask=mpeak ).item()
        total_brier_acc   += brier_score(p_acc,   acc,   mask=macc  ).item()

        nb += 1

        # Median quantile
        q50_log = q_hat[..., k50]
        q50_mm  = torch.expm1(q50_log).clamp_min(0.0)

        B, H, N = y_mm.shape
        if H_out is None:
            H_out = H
            sum_abs = torch.zeros(H_out, device="cpu")
            sum_sq  = torch.zeros(H_out, device="cpu")
            count   = torch.zeros(H_out, device="cpu")

        for h in range(H):
            m = (my[:, h, :] > 0.5)
            if m.any():
                err = (q50_mm[:, h, :][m] - y_mm[:, h, :][m]).detach().cpu()
                sum_abs[h] += err.abs().sum()
                sum_sq[h]  += (err**2).sum()
                count[h]   += m.sum().detach().cpu()

        flash_p_list.append(p_flash.detach().cpu().reshape(-1))
        flash_y_list.append(flash.detach().cpu().reshape(-1))
        flash_m_list.append(mflash.detach().cpu().reshape(-1))

        peak_p_list.append(p_peak.detach().cpu().reshape(-1))
        peak_y_list.append(peak.detach().cpu().reshape(-1))
        peak_m_list.append(mpeak.detach().cpu().reshape(-1))

        acc_p_list.append(p_acc.detach().cpu().reshape(-1))
        acc_y_list.append(acc.detach().cpu().reshape(-1))
        acc_m_list.append(macc.detach().cpu().reshape(-1))

    out = {
        "CRPS_log":   total_crps_log / max(nb, 1),
        "CRPS_mm":    total_crps_mm  / max(nb, 1),
        "Brier_flash": total_brier_flash / max(nb, 1),
        "Brier_peak":  total_brier_peak  / max(nb, 1),
        "Brier_acc":   total_brier_acc   / max(nb, 1),
    }

    # MAE / RMSE by lead
    maes, rmses, counts = [], [], []
    for h in range(H_out):
        if count[h].item() < 1:
            maes.append(np.nan); rmses.append(np.nan); counts.append(0)
        else:
            maes.append(float((sum_abs[h] / count[h]).item()))
            rmses.append(float(torch.sqrt(sum_sq[h] / count[h]).item()))
            counts.append(int(count[h].item()))
    out.update({"MAE_mm_by_lead": maes, "RMSE_mm_by_lead": rmses, "n_valid_by_lead": counts})

    # Event metrics
    flash_p = torch.cat(flash_p_list).numpy()
    flash_y = torch.cat(flash_y_list).numpy()
    flash_m = torch.cat(flash_m_list).numpy()

    peak_p  = torch.cat(peak_p_list).numpy()
    peak_y  = torch.cat(peak_y_list).numpy()
    peak_m  = torch.cat(peak_m_list).numpy()

    acc_p   = torch.cat(acc_p_list).numpy()
    acc_y   = torch.cat(acc_y_list).numpy()
    acc_m   = torch.cat(acc_m_list).numpy()

    out["Flash_metrics"]  = event_metrics_binary(flash_p, flash_y, flash_m, thresholds=thresholds)
    out["Peak24_metrics"] = event_metrics_binary(peak_p,  peak_y,  peak_m,  thresholds=thresholds)
    out["Acc24_metrics"]  = event_metrics_binary(acc_p,   acc_y,   acc_m,   thresholds=thresholds)

    return out

# ============================================================
# 6) LOAD FINAL MODELS + REBUILD TEST SET + EVALUATE
# ============================================================

print("\nLoading final LightGBM artifacts from:", FINAL_MODELS_PATH)
art = joblib.load(FINAL_MODELS_PATH)

quantiles   = tuple(art["quantiles"])
lgbm_reg    = art["lgbm_reg"]
lgbm_flash  = art["lgbm_flash"]
lgbm_peak   = art["lgbm_peak"]
lgbm_acc    = art["lgbm_acc"]
scaler_mean = np.array(art["scaler_mean"])
scaler_std  = np.array(art["scaler_std"])
thr3h       = np.array(art["thr3h"])
thrAcc24    = np.array(art["thrAcc24"])
T_in        = int(art["T_in"])
H_out       = int(art["H_out"])
train_frac  = float(art["train_frac"])
val_frac    = float(art["val_frac"])

print("Loaded quantiles  :", quantiles)
print("T_in, H_out       :", T_in, H_out)
print("train_frac, val_frac :", train_frac, val_frac)

# Rebuild preprocessing from df_rain
prep = prepare_full_grid(df_rain)
stations = prep["stations"]
N_stations = len(stations)

times = prep["times"]
T_total = len(times)

train_end, val_end = make_splits(T_total, train_frac=train_frac, val_frac=val_frac)
print("\nSplit indices (recomputed):")
print("  Train: 0 →", train_end)
print("  Val:   ", train_end, "→", val_end)
print("  Test:  ", val_end, "→", T_total)

# Rebuild scaler from saved stats (NO refit)
scaler = NaNIgnoringStandardScaler()
scaler.mean_ = scaler_mean
scaler.std_  = scaler_std

X_raw = prep["X_raw"]
T_, N_, F_in_raw = X_raw.shape
X_flat = X_raw.reshape(T_ * N_, F_in_raw)
X_scaled_flat = scaler.transform(X_flat)
X_scaled = X_scaled_flat.reshape(T_, N_, F_in_raw).astype(np.float32)

# Acc24 + MaskAcc24 recomputed (data-dependent only)
Acc24, MaskAcc24 = compute_acc24(prep["Y_rain"], prep["M_y"], H_out=H_out)

# Build TEST dataset only, using SAVED thresholds
ds_test = BanglaRainDataset(
    X_scaled, prep["M_in"], prep["time_feats"],
    prep["Y_rain"], prep["M_y"],
    Acc24, MaskAcc24,
    prep["season_ids"],
    thr3h, thrAcc24,
    t_start=val_end, t_end=T_total,
    T_in=T_in, H_out=H_out,
)

test_loader = make_loaders(ds_test, batch_size=256)
print("Test dataset size:", len(ds_test))

# Wrap LightGBM models in PyTorch-like module
model = StationWiseLGBMBaseline(
    lgbm_regressors=lgbm_reg,
    lgbm_flash=lgbm_flash,
    lgbm_peak=lgbm_peak,
    lgbm_acc=lgbm_acc,
    num_stations=N_stations,
    H_out=H_out,
    quantiles=quantiles,
).to(device)

print("\nModel wrapper built. Evaluating on TEST split...")

with torch.no_grad():
    test_scores = evaluate(
        model,
        test_loader,
        quantiles=quantiles,
    )

# ============================================================
# 7) PRINT TEST METRICS
# ============================================================

print("\n========== LIGHTGBM STATION-WISE TEST METRICS ==========")
print(f"CRPS_log   : {test_scores['CRPS_log']:.4f}")
print(f"CRPS_mm    : {test_scores['CRPS_mm']:.4f}")
print(f"Brier_flash: {test_scores['Brier_flash']:.4f}")
print(f"Brier_peak : {test_scores['Brier_peak']:.4f}")
print(f"Brier_acc  : {test_scores['Brier_acc']:.4f}")

fm = test_scores["Flash_metrics"]
pm = test_scores["Peak24_metrics"]
am = test_scores["Acc24_metrics"]

print("\n[Flash 3h]  PR-AUC={:.4f}, ROC-AUC={:.4f}, n_valid={}".format(fm["pr_auc"], fm["roc_auc"], fm["n_valid"]))
print("[Peak24]   PR-AUC={:.4f}, ROC-AUC={:.4f}, n_valid={}".format(pm["pr_auc"], pm["roc_auc"], pm["n_valid"]))
print("[Acc24]    PR-AUC={:.4f}, ROC-AUC={:.4f}, n_valid={}".format(am["pr_auc"], am["roc_auc"], am["n_valid"]))

print("\nMAE_mm_by_lead :", test_scores['MAE_mm_by_lead'])
print("RMSE_mm_by_lead:", test_scores['RMSE_mm_by_lead'])
print("n_valid_by_lead:", test_scores['n_valid_by_lead'])


Using device: cpu
Final models path : C:\Users\pc\Documents\Rainfall\Baselines\LightGBM\Final_Run\v1\final_lightgbm_models.pkl

Loading final LightGBM artifacts from: C:\Users\pc\Documents\Rainfall\Baselines\LightGBM\Final_Run\v1\final_lightgbm_models.pkl
Loaded quantiles  : (0.1, 0.5, 0.9)
T_in, H_out       : 16 8
train_frac, val_frac : 0.7 0.15
Total stations: 34
Total unique times: 61360

Split indices (recomputed):
  Train: 0 → 42952
  Val:    42952 → 52156
  Test:   52156 → 61360
Test dataset size: 9181

Model wrapper built. Evaluating on TEST split...


C:\Users\pc\AppData\Local\Temp\ipykernel_9604\884182916.py:313: RuntimeWarning: All-NaN slice encountered
  y_peak = np.nanmax(np.where(m_win > 0.5, y_win, np.nan), axis=0)



========== LIGHTGBM STATION-WISE TEST METRICS ==========
CRPS_log   : 0.1172
CRPS_mm    : 0.6098
Brier_flash: 0.0338
Brier_peak : 0.1127
Brier_acc  : 0.0367

[Flash 3h]  PR-AUC=0.3472, ROC-AUC=0.8886, n_valid=307503
[Peak24]   PR-AUC=0.5547, ROC-AUC=0.8561, n_valid=307535
[Acc24]    PR-AUC=0.2020, ROC-AUC=0.8447, n_valid=307195

MAE_mm_by_lead : [0.6631738543510437, 0.6679536700248718, 0.6700106859207153, 0.671186089515686, 0.6719302535057068, 0.6723476052284241, 0.6726981401443481, 0.6726735234260559]
RMSE_mm_by_lead: [3.98486328125, 3.994307279586792, 3.9979701042175293, 4.000021457672119, 4.000892639160156, 4.001763343811035, 4.001806259155273, 4.002007484436035]
n_valid_by_lead: [307503, 307500, 307497, 307494, 307491, 307488, 307485, 307482]
